# Session 3: フィードバック制御入門
# Session 3: Introduction to Feedback Control

## 目標 / Objective

P 制御で 1 次系の位置制御を体験し、フィードバックの概念を理解する。

Experience position control of a first-order system with P control and understand the concept of feedback.

## この Notebook で学ぶこと / What You'll Learn

| トピック | 説明 |
|---------|------|
| 開ループ vs 閉ループ | フィードバックの効果 |
| P 制御 | 比例ゲインと応答の関係 |
| 定常偏差 | P 制御の限界 |
| 1 次系の応答 | 時定数と立ち上がり時間 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from stampfly_edu.sim import FirstOrderPlant, simulate_pid, compare_controllers
from stampfly_edu.sim.simulate import compute_metrics

print("Ready! / 準備完了！")

## 1. 開ループ制御の問題 / Open-Loop Control Problems

### 理論 / Theory

**開ループ制御**（フィードバックなし）:

$$u(t) = u_{\text{fixed}}$$

入力を一定にすると、プラントの特性が変わった場合に対応できません。

**1次系のプラントモデル**:

$$G(s) = \frac{K}{\tau s + 1}$$

- $K$: DCゲイン（入力に対する最終出力の比）
- $\tau$: 時定数（応答の速さを決める）

In [ ]:
# Open-loop simulation: constant input to first-order plant
# 開ループシミュレーション: 1次系に一定入力

plant = FirstOrderPlant(K=1.0, tau=1.0)
dt = 0.01
duration = 10.0
n = int(duration / dt)

t = np.arange(n) * dt
y_openloop = np.zeros(n)
u_openloop = np.ones(n)  # Constant input = 1.0

for i in range(n):
    y_openloop[i] = plant.step(u_openloop[i], dt)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(t, y_openloop, 'b-', linewidth=1.5, label='Output / 出力')
ax1.axhline(y=1.0, color='r', linestyle='--', label='Target / 目標')
ax1.set_ylabel('Output / 出力')
ax1.set_title('Open-Loop Step Response / 開ループステップ応答')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(t, u_openloop, 'g-', linewidth=1.5)
ax2.set_ylabel('Input / 入力')
ax2.set_xlabel('Time (s) / 時間')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final output / 最終出力: {y_openloop[-1]:.4f}")
print(f"Time constant tau / 時定数: {plant.tau} s")
print(f"63.2% response time / 63.2%応答時間: {t[np.argmax(y_openloop >= 0.632)]:.2f} s")

## 2. 閉ループ制御 (P制御) / Closed-Loop Control (P Control)

### 理論 / Theory

**P制御**（比例制御）:

$$u(t) = K_p \cdot e(t) = K_p \cdot (r(t) - y(t))$$

ブロック線図 / Block Diagram:
```
r(t) ──→ [+] ──→ [Kp] ──→ [Plant] ──→ y(t)
          ↑-                             │
          └─────────────────────────────┘
```

- $r(t)$: 目標値 / setpoint
- $e(t)$: 偏差 / error
- $K_p$: 比例ゲイン / proportional gain
- $y(t)$: 出力 / output

In [ ]:
# P control with different gains
# 異なるゲインでの P 制御
plant = FirstOrderPlant(K=1.0, tau=1.0)

params_list = [
    {"Kp": 0.5},
    {"Kp": 1.0},
    {"Kp": 2.0},
    {"Kp": 5.0},
]
labels = ["Kp=0.5", "Kp=1.0", "Kp=2.0", "Kp=5.0"]

fig = compare_controllers(
    plant, params_list, labels=labels,
    title="P Control: Effect of Kp / P制御: Kp の効果",
    duration=10.0,
)
plt.show()

### 考察 / Discussion

上のグラフから読み取れること：

1. **Kp を大きくすると**：応答が速くなる
2. **P制御の限界**：目標値に完全には到達しない（**定常偏差**）

定常偏差の式（1次系の場合）:

$$e_{ss} = \frac{1}{1 + K_p \cdot K}$$

In [ ]:
# Verify steady-state error formula
# 定常偏差の公式を検証
print("Kp | Predicted SS Error | Actual SS Error")
print("-" * 50)

K_plant = 1.0
for params, label in zip(params_list, labels):
    Kp = params["Kp"]
    predicted = 1.0 / (1.0 + Kp * K_plant)
    
    result = simulate_pid(plant, Kp=Kp, duration=20.0)
    actual = abs(1.0 - result["output"].iloc[-1])
    
    print(f"{label:8s} | {predicted:.4f}             | {actual:.4f}")

## 3. PI 制御で定常偏差を消す / Eliminating SS Error with PI Control

### 理論 / Theory

積分項 (I) を加えることで定常偏差を 0 にできます：

$$u(t) = K_p \left( e(t) + \frac{1}{T_i} \int_0^t e(\tau) d\tau \right)$$

- $T_i$: 積分時間 / integral time
- $T_i$ が小さいほど積分の効きが強い

In [ ]:
# P vs PI comparison
# P制御 vs PI制御の比較
plant = FirstOrderPlant(K=1.0, tau=1.0)

params_list = [
    {"Kp": 2.0, "Ti": 0.0},   # P only
    {"Kp": 2.0, "Ti": 2.0},   # PI (slow integral)
    {"Kp": 2.0, "Ti": 0.5},   # PI (fast integral)
]
labels = ["P only (Ti=0)", "PI (Ti=2.0)", "PI (Ti=0.5)"]

fig = compare_controllers(
    plant, params_list, labels=labels,
    title="P vs PI Control / P制御 vs PI制御",
    duration=10.0,
)
plt.show()

## 4. 外乱への応答 / Response to Disturbances

フィードバック制御の重要な利点は外乱を抑制できることです。

In [ ]:
# Disturbance rejection comparison
# 外乱抑制の比較
from stampfly_edu.sim.simulate import simulate_pid

plant = FirstOrderPlant(K=1.0, tau=1.0)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

for Kp, color, label in [(0.5, 'b', 'Kp=0.5'), (2.0, 'r', 'Kp=2.0'), (5.0, 'g', 'Kp=5.0')]:
    result = simulate_pid(
        plant, Kp=Kp, Ti=1.0,
        setpoint=1.0, duration=15.0,
        disturbance_time=7.0, disturbance_value=0.5,
    )
    axes[0].plot(result['time'], result['output'], color=color, label=label)
    axes[1].plot(result['time'], result['control'], color=color, label=label)

axes[0].axhline(y=1.0, color='k', linestyle='--', linewidth=0.5)
axes[0].axvline(x=7.0, color='gray', linestyle=':', label='Disturbance / 外乱')
axes[0].set_ylabel('Output / 出力')
axes[0].set_title('Disturbance Rejection / 外乱抑制')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel('Control / 制御入力')
axes[1].set_xlabel('Time (s) / 時間')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 考察課題 / Discussion Questions

1. P制御でゲインを無限大にしたら定常偏差はゼロになるか？実際には何が問題になるか？
2. PI制御で Ti を小さくしすぎるとどうなるか？シミュレーションで確認してみよう。
3. ドローンの高度制御に P 制御だけ使った場合、どのような問題が起きるか？

---

1. If we increase P-gain to infinity, would SS error become zero? What problem arises?
2. What happens if Ti is made too small in PI control? Verify with simulation.
3. What problems arise when using only P control for drone altitude control?

In [ ]:
# Your experiments here / ここで実験してみてください
